# Notebook 02 - Preprocessing et Embeddings SBERT

**Projet** : MixCraft - M1 Data Engineering & IA, EFREI Paris  
**Auteurs** : Adam Beloucif & Emilien Morice - Janvier 2026

## Objectifs

1. Nettoyer et normaliser les textes du corpus
2. Calculer les embeddings SBERT (all-MiniLM-L6-v2, 384 dims)
3. Visualiser l'espace semantique via t-SNE
4. Clustering KMeans pour identifier les familles de cocktails

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path

EFREI_NAVY = '#163767'
EFREI_PINK = '#FF43B8'
EFREI_BLUE = '#0C78B4'

# Chargement corpus
corpus_path = Path('../data/processed/corpus_cocktails.csv')
if corpus_path.exists():
    df = pd.read_csv(corpus_path)
else:
    from src.data_loader import load_all_datasets
    df = load_all_datasets()

print(f'Corpus charge : {len(df)} cocktails')

## 1. Preprocessing du texte

In [ ]:
import re

def preprocess_text(text: str) -> str:
    """Nettoyage de base : suppression des caracteres speciaux, normalisation des espaces."""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'\s+', ' ', text)         # espaces multiples
    text = re.sub(r'[^a-zA-Z0-9 ,.!?\-]', '', text)  # caracteres speciaux
    return text.strip().lower()

def normalize_ingredients(ing_str: str) -> str:
    """Normalise la liste des ingredients : supprime les quantites, met en minuscules."""
    if not isinstance(ing_str, str):
        return ''
    # Supprime les mesures (60ml, 2 oz, 1/2 cup...)
    cleaned = re.sub(r'\d+(?:\.\d+)?\s*(?:ml|cl|oz|tsp|tbsp|cup|dash)?', '', ing_str, flags=re.IGNORECASE)
    # Normalise les separateurs
    parts = [p.strip().lower() for p in re.split(r'[,\n]', cleaned) if p.strip()]
    return ', '.join(parts)


# Application
df['instructions_clean'] = df['instructions'].apply(preprocess_text)
df['ingredients_clean'] = df['ingredients'].apply(normalize_ingredients)

# Construction du texte complet pour l'embedding
df['text_embedding'] = (
    df['name'].fillna('') + '. '
    + 'Category: ' + df['category'].fillna('') + '. '
    + 'Ingredients: ' + df['ingredients_clean'] + '. '
    + df['instructions_clean']
)

print('Exemple de texte pour embedding :')
print(df['text_embedding'].iloc[0][:300])

## 2. Calcul des embeddings SBERT

In [ ]:
from src.embeddings import EmbeddingEngine

engine = EmbeddingEngine(model_name='all-MiniLM-L6-v2', cache=True)

print(f'Modele : {engine.model_name}')
print(f'Dimension des embeddings : {engine.embedding_dim}')

texts = df['text_embedding'].tolist()
embeddings = engine.encode(texts, batch_size=64, show_progress=True)

print(f'\nMatrice d\'embeddings : {embeddings.shape}')
print(f'Norme L2 (premier vecteur) : {np.linalg.norm(embeddings[0]):.4f}')  # doit etre ~1.0 (normalise)

In [ ]:
# Sauvegarde des embeddings
import numpy as np
np.save('../data/processed/embeddings.npy', embeddings)
print('Embeddings sauvegardes : data/processed/embeddings.npy')

## 3. Visualisation t-SNE

In [ ]:
from sklearn.manifold import TSNE

# t-SNE en 2D (perplexite = 30 par defaut, recommande pour 300-1000 pts)
print('Calcul t-SNE (peut prendre 1-2 min)...')
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate=200,
    n_iter=1000,
    random_state=42,
    init='pca',
)
embeddings_2d = tsne.fit_transform(embeddings)

df['tsne_x'] = embeddings_2d[:, 0]
df['tsne_y'] = embeddings_2d[:, 1]

print(f't-SNE termine. Forme : {embeddings_2d.shape}')

In [ ]:
# Visualisation t-SNE coloriee par categorie
top_categories = df['category'].value_counts().head(6).index.tolist()
palette = [EFREI_NAVY, EFREI_BLUE, EFREI_PINK, '#2da8a8', '#e8a020', '#8B4513']
cat_color_map = {cat: palette[i] for i, cat in enumerate(top_categories)}

fig, ax = plt.subplots(figsize=(13, 8))

for cat, color in cat_color_map.items():
    mask = df['category'] == cat
    ax.scatter(
        df.loc[mask, 'tsne_x'], df.loc[mask, 'tsne_y'],
        c=color, alpha=0.6, s=25, label=cat, edgecolors='none'
    )

# Autres categories
other_mask = ~df['category'].isin(top_categories)
ax.scatter(df.loc[other_mask, 'tsne_x'], df.loc[other_mask, 'tsne_y'],
           c='#cccccc', alpha=0.3, s=15, label='Autre')

ax.set_title('Espace semantique des cocktails (t-SNE, SBERT 384d)', fontsize=15, fontweight='bold', color=EFREI_NAVY)
ax.set_xlabel('t-SNE dimension 1')
ax.set_ylabel('t-SNE dimension 2')
ax.legend(loc='upper right', fontsize=9, framealpha=0.8)
ax.set_facecolor('#f8f9fc')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('../assets/fig_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Clustering KMeans

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Recherche du nombre optimal de clusters via la methode du coude
inertias = []
sil_scores = []
k_range = range(3, 16)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(embeddings, labels, sample_size=500, random_state=42))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertias, 'o-', color=EFREI_NAVY, linewidth=2, markersize=6)
axes[0].set_title('Methode du coude (Inertie)', fontweight='bold', color=EFREI_NAVY)
axes[0].set_xlabel('Nombre de clusters K')
axes[0].set_ylabel('Inertie')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_range), sil_scores, 's-', color=EFREI_PINK, linewidth=2, markersize=6)
best_k = list(k_range)[np.argmax(sil_scores)]
axes[1].axvline(best_k, color=EFREI_NAVY, linestyle='--', alpha=0.7, label=f'Optimal K={best_k}')
axes[1].set_title('Score de silhouette', fontweight='bold', color=EFREI_NAVY)
axes[1].set_xlabel('Nombre de clusters K')
axes[1].set_ylabel('Score silhouette')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../assets/fig_kmeans_elbow.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'K optimal (silhouette max) : {best_k}')

In [ ]:
# KMeans avec K optimal
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(embeddings)

# Noms des clusters (top ingredient par cluster)
cluster_labels = {}
for c in range(best_k):
    mask = df['cluster'] == c
    top_cat = df.loc[mask, 'category'].mode().iloc[0] if mask.sum() > 0 else f'Cluster {c}'
    cluster_labels[c] = f'C{c}: {top_cat}'

# Visualisation t-SNE avec clusters
fig, ax = plt.subplots(figsize=(13, 8))
palette_k = cm.get_cmap('tab20', best_k)

for c in range(best_k):
    mask = df['cluster'] == c
    ax.scatter(
        df.loc[mask, 'tsne_x'], df.loc[mask, 'tsne_y'],
        c=[palette_k(c / best_k)], alpha=0.65, s=25,
        label=cluster_labels[c], edgecolors='none'
    )

ax.set_title(f't-SNE avec K={best_k} clusters (KMeans, embeddings SBERT)', fontsize=14, fontweight='bold', color=EFREI_NAVY)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(loc='upper right', fontsize=8, framealpha=0.85, ncol=2)
ax.set_facecolor('#f8f9fc')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('../assets/fig_tsne_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sauvegarde corpus enrichi avec clusters
df.to_csv('../data/processed/corpus_avec_clusters.csv', index=False)
print('Corpus enrichi sauvegarde.')

print('\n=== Resume Embeddings ===')
print(f'  Modele SBERT         : {engine.model_name}')
print(f'  Dimension            : {embeddings.shape[1]}')
print(f'  Cocktails encodes    : {embeddings.shape[0]}')
print(f'  Clusters trouves     : {best_k}')
print(f'  Score silhouette max : {max(sil_scores):.3f}')